In [4]:
import sys
import os
import pandas as pd

sys.path.insert(0, os.path.abspath('..'))
import random_no_generation
random_no_generation.CSV_FILE = os.path.abspath('../randomno.csv')

from random_no_generation import (
    clean_index,
    return_ran,
    return_ran_array,
    random_normal,
    random_normal_from_variance,
    random_normal_adv,
    random_normal_adv_from_variance,
)

In [5]:
df = pd.read_csv('./csv_input/stage6_synced_polygons.csv')
df.columns

Index(['County', 'Shape_ID', 'geometry', 'Area', 'Population_Density',
       'Region', 'Location', 'County_Type'],
      dtype='object')

In [6]:
raw = return_ran_array(len(df))
df['Capacity'] = [round(0.85 + (r - 1) / (10_000_000 - 1) * 0.15, 2) for r in raw]


In [ ]:
exp_popu = [round(row['Capacity'] * row['Population_Density'] * row['Area'])
            for _, row in df.iterrows()]

input_sd = [0.05 * m for m in exp_popu]

df['Population'] = random_normal_adv(exp_popu, input_sd)

,Shape_ID,Area,Population_Density,Capacity,Population
0,NC1_001,0.2819,11245,0.99,3470
1,NC1_002,0.4191,8520,0.99,3231
2,NC1_003,0.3371,8205,0.97,2656
3,NC1_004,0.4594,8210,0.93,3708
4,NC1_005,0.2069,9255,0.90,1787
5,NC1_006,0.7160,7800,0.98,5562
6,NC1_007,0.2716,10645,0.93,2500
7,NC1_008,0.5031,7355,0.91,3412
8,NC1_009,0.0911,15360,0.86,1242
9,NC1_010,2.6123,3545,0.86,8433


In [30]:
DEMO_COLS = ['S', 'A1', 'A2', 'B1', 'B2', 'C1', 'C2', 'D1', 'D2', 'E1', 'E2', 'F1', 'F2']

community_df = pd.read_csv('./csv_input/Info_CSV/Community_matrix.csv')

pop_df = pd.read_csv("inspect_pop_df.csv")

pop_df = pop_df.merge(
    community_df.loc[community_df['Region'] == 'Capital', ['Type', 'Code'] + DEMO_COLS],
    left_on=['Location', 'County_Type'],
    right_on=['Type', 'Code'],
    how='left'
).drop(columns=['Type', 'Code'])

for col in DEMO_COLS:
    pop_df[col] = (pop_df[col] / 100 * pop_df['Population']).round().fillna(0).astype(int)

In [31]:
df_county = pop_df.copy()
df_county['County'] = df_county['Shape_ID'].str.split('_').str[0]

df_county = (
    df_county.groupby('County')[['Population'] + DEMO_COLS]
    .sum()
    .reset_index()
)



In [32]:
df_district = pop_df.copy()
df_district['District'] = df_district['Shape_ID'].str.extract(r'^([A-Za-z]+)')

df_district = (
    df_district.groupby('District')[['Population'] + DEMO_COLS]
    .sum()
    .reset_index()
)

df_district


,District,Population,S,A1,A2,B1,B2,C1,C2,D1,D2,E1,E2,F1,F2
0,E,2771484,30114,49768,65179,183072,60025,218398,180509,854612,74670,365103,149346,425824,103371
1,NC,1418409,24559,39811,41589,188540,102046,131837,120879,278566,39647,156617,75760,161526,55425
2,W,416005,5829,13206,8922,43710,12867,31054,27810,68878,5887,48752,18162,64666,66263
3,WL,529692,5295,12372,9529,30327,8056,50127,38877,105985,6118,81207,19766,86994,75040


In [33]:
#df_county.to_csv('inspect_county.csv', index=False)
df_district.to_csv('inspect_district2.csv', index=False)
#pop_df.to_csv('inspect_pop_df.csv', index=False)

In [38]:
df.to_csv('./csv_input/stage6_synced_polygons_updated.csv', index=False)